# 👍 제스처 인식 (Gesture Recognition)

MediaPipe의 **GestureRecognizer**를 사용하여 손 모양으로 제스처를 인식합니다.

```bash
pip install mediapipe opencv-python numpy matplotlib
```

| 항목 | 내용 |
|------|------|
| **태스크** | GestureRecognizer |
| **모델 파일** | `models/gesture_recognizer.task` (~8.0 MB) |
| **입력** | 이미지 / 비디오 / 실시간 스트림 |
| **출력** | 인식된 제스처 이름 + 신뢰도 + 손 랜드마크 |
| **인식 가능 제스처** | 7가지 (Thumb_Up, Thumb_Down, Victory, …) |

> MediaPipe 1.0 Tasks API를 사용합니다. 예전 `mp.solutions.gesture_recognizer`는 더 이상 사용되지 않습니다.


---
## 1. GestureRecognizer 설정

GestureRecognizer는 손 랜드마크를 검출한 뒤, 손가락 관절의 상대적 위치를 분석해 **미리 정의된 7가지 제스처** 중 가장 확률이 높은 것을 반환합니다.

**구성 단계:**
1. `BaseOptions`에 모델 파일 경로 지정
2. `GestureRecognizerOptions` 생성 (실행 모드, 손 개수 등)
3. `GestureRecognizer.create_from_options()`로 인식기 객체 생성
4. `recognizer.recognize(mp_image)`로 추론 실행


In [ ]:
# 제스처 인식기 기본 설정
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import os

# 모델 파일 경로
MODEL_PATH = 'models/gesture_recognizer.task'

# 모델 파일 존재 확인
if os.path.exists(MODEL_PATH):
    size_mb = os.path.getsize(MODEL_PATH) / 1024 / 1024
    print(f'✓ 모델 파일 확인: {MODEL_PATH} ({size_mb:.1f} MB)')
else:
    print(f'✗ 모델 파일 없음: {MODEL_PATH}')
    print('  models/gesture_recognizer.task 파일을 다운로드하세요.')
    print('  https://storage.googleapis.com/mediapipe-models/gesture_recognizer/gesture_recognizer/float16/1/gesture_recognizer.task')

# 1. BaseOptions 설정
base_options = python.BaseOptions(
    model_asset_path=MODEL_PATH
)

# 2. GestureRecognizerOptions 설정
options = vision.GestureRecognizerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.IMAGE,
    num_hands=2,
    min_hand_detection_confidence=0.5,
    min_hand_presence_confidence=0.5,
    min_tracking_confidence=0.5
)

# 3. 인식기 객체 생성
recognizer = vision.GestureRecognizer.create_from_options(options)
print(f'✓ GestureRecognizer 생성 완료: {type(recognizer).__name__}')
print(f'  실행 모드: IMAGE')
print(f'  최대 손 개수: {options.num_hands}')


---
## 2. 인식 가능한 제스처 목록

`gesture_recognizer.task` 모델은 **7가지 제스처**를 인식할 수 있습니다.

| 제스처 이름 | 의미 | 손 모양 |
|------------|------|---------|
| `None` | 인식 불가 / 알 수 없음 | 미정의 |
| `Closed_Fist` | 주먹 | ✊ |
| `Open_Palm` | 편 손바닥 | 🖐️ |
| `Pointing_Up` | 검지 위로 | ☝️ |
| `Thumb_Down` | 엄지 아래 | 👎 |
| `Thumb_Up` | 엄지 위 | 👍 |
| `Victory` | V (승리) | ✌️ |
| `ILoveYou` | I Love You (새끼·엄지·새끼손가락 펴기) | 🤟 |

> 제스처 이름은 **영어 Pascal_Snake_Case**로 반환됩니다.
> 한국어 라벨로 변환하려면 별도 매핑 딕셔너리를 사용합니다 (다음 셀 참고).


In [ ]:
# 인식 가능한 제스처 목록 (모델이 반환하는 7가지 + None)
GESTURES = [
    'None',
    'Closed_Fist',
    'Open_Palm',
    'Pointing_Up',
    'Thumb_Down',
    'Thumb_Up',
    'Victory',
    'ILoveYou',
]

# 한국어 라벨 매핑 (시각화/제어에 사용)
GESTURE_KR = {
    'None': '알 수 없음',
    'Closed_Fist': '주먹 ✊',
    'Open_Palm': '손바닥 🖐️',
    'Pointing_Up': '검지 ☝️',
    'Thumb_Down': '엄지 아래 👎',
    'Thumb_Up': '엄지 위 👍',
    'Victory': '승리 ✌️',
    'ILoveYou': 'I Love You 🤟',
}

# 이모지 매핑 (화면 표시용)
GESTURE_EMOJI = {
    'None': '❔',
    'Closed_Fist': '✊',
    'Open_Palm': '🖐️',
    'Pointing_Up': '☝️',
    'Thumb_Down': '👎',
    'Thumb_Up': '👍',
    'Victory': '✌️',
    'ILoveYou': '🤟',
}

print('Gesture Recognizer가 인식할 수 있는 제스처:')
print('-' * 40)
for g in GESTURES:
    print(f'  {GESTURE_EMOJI[g]:<3} {g:<14} → {GESTURE_KR[g]}')
print('-' * 40)
print(f'총 {len(GESTURES)-1}가지 제스처 (+ None)')

# 이모지 매핑은 모델에서 제공하지 않으므로 직접 정의한다.
# 모델은 gesture name(str)과 confidence(float)만 반환한다.


---
## 3. 정적 이미지 제스처 인식

단일 이미지에서 제스처를 인식해 봅니다.
실제 손 사진이 없으므로, OpenCV로 **간단한 손 모양 템플릿**을 그려 동작을 확인합니다.

> 실제 제스처 인식을 위해서는 손이 포함된 사진을 사용하세요.
> 템플릿 이미지는 랜드마크 검출이 어려워 `None`이 반환될 수 있습니다.
> 코드 구조를 이해하는 데 집중하세요.


In [ ]:
# 정적 이미지 제스처 인식
import numpy as np
import cv2
import mediapipe as mp

# MediaPipe Image 변환 헬퍼
def numpy_to_mp_image(np_image):
    """NumPy BGR 배열 → MediaPipe SRGB Image"""
    # MediaPipe는 RGB 순서를 기대하므로 BGR→RGB 변환
    rgb = cv2.cvtColor(np_image, cv2.COLOR_BGR2RGB)
    return mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=rgb
    )

# 템플릿 이미지 생성 (밝은 배경에 단순한 손 모양)
def make_hand_template(shape='palm', size=400):
    """간단한 손 모양 템플릿 생성 (실제 인식은 어려움, 구조 학습용)"""
    img = np.full((size, size, 3), 240, dtype=np.uint8)  # 밝은 배경
    cx, cy = size // 2, size // 2

    if shape == 'palm':
        # 손바닥: 중앙에 타원
        cv2.ellipse(img, (cx, cy), (60, 100), 0, 0, 360, (180, 120, 80), -1)
        # 손가락 5개
        for i, dx in enumerate([-60, -30, 0, 30, 60]):
            cv2.line(img, (cx+dx, cy-90), (cx+dx, cy-160), (180, 120, 80), 22)
    elif shape == 'victory':
        cv2.ellipse(img, (cx, cy), (60, 100), 0, 0, 360, (180, 120, 80), -1)
        # 검지 + 중지만
        cv2.line(img, (cx-20, cy-90), (cx-40, cy-170), (180, 120, 80), 22)
        cv2.line(img, (cx+20, cy-90), (cx+40, cy-170), (180, 120, 80), 22)
    elif shape == 'thumb_up':
        cv2.ellipse(img, (cx, cy), (60, 80), 0, 0, 360, (180, 120, 80), -1)
        # 엄지만 위로
        cv2.line(img, (cx-80, cy-40), (cx-80, cy-150), (180, 120, 80), 26)
    return img

# 3가지 템플릿에 대해 제스처 인식 시도
templates = {
    'Open_Palm (손바닥)': make_hand_template('palm'),
    'Victory (승리)':     make_hand_template('victory'),
    'Thumb_Up (엄지 위)': make_hand_template('thumb_up'),
}

print('정적 이미지 제스처 인식 결과:')
print('=' * 60)
for label, img in templates.items():
    mp_img = numpy_to_mp_image(img)

    # recognize() 호출 — IMAGE 모드
    result = recognizer.recognize(mp_img)

    # 결과 추출
    # result.gestures: List[List[Category]]  (손별 제스처 리스트)
    # result.handedness: List[List[Category]] (손별 왼/오른손)
    # result.hand_landmarks: List[List[NormalizedLandmark]]
    n_hands = len(result.hand_landmarks)

    if n_hands > 0:
        top_gesture = result.gestures[0][0]  # 첫 번째 손의 top-1 제스처
        gesture_name = top_gesture.category_name
        score = top_gesture.score
        handed = result.handedness[0][0].category_name
        print(f'  [{label}]')
        print(f'    검출된 손: {n_hands}개 ({handed})')
        print(f'    제스처: {gesture_name} ({GESTURE_KR.get(gesture_name, gesture_name)})')
        print(f'    신뢰도: {score:.3f}')
    else:
        print(f'  [{label}]')
        print(f'    검출된 손: 없음 (템플릿 이미지는 랜드마크 검출이 어려움)')
        print(f'    → 실제 손 사진을 사용하면 제스처가 인식됩니다.')
    print('-' * 60)

# GestureRecognizerResult 구조 확인
print('\nGestureRecognizerResult 속성:')
print('  result.gestures       :', type(result.gestures).__name__)
print('  result.handedness     :', type(result.handedness).__name__)
print('  result.hand_landmarks :', type(result.hand_landmarks).__name__)
print('  result.hand_world_landmarks :', type(result.hand_world_landmarks).__name__)


---
## 4. 결과 객체 구조 상세

`recognizer.recognize()`가 반환하는 `GestureRecognizerResult`의 구조를 살펴봅니다.

```
GestureRecognizerResult
├── gestures: List[List[Category]]          # 손별 제스처 후보 리스트
│   └── gestures[0][0]                      # 첫 번째 손의 top-1 제스처
│       ├── category_name: str              # "Victory"
│       ├── score: float                    # 0.0 ~ 1.0
│       └── index: int                      # 카테고리 인덱스
├── handedness: List[List[Category]]        # 손별 왼/오른손
│   └── handedness[0][0].category_name      # "Left" / "Right"
├── hand_landmarks: List[List[NormalizedLandmark]]   # 21개 관절 (정규화 좌표)
└── hand_world_landmarks: List[List[Landmark]]        # 21개 관절 (실세계 좌표, cm)
```

> 손이 2개 검출되면 각 리스트의 길이는 2가 됩니다.


In [ ]:
# 결과 구조 유틸 함수 (이후 셀에서 재사용)
import mediapipe as mp

def format_gesture_result(result):
    """GestureRecognizerResult를 읽기 쉽게 포맷팅"""
    lines = []
    n = len(result.hand_landmarks)
    lines.append(f'검출된 손: {n}개')
    for i in range(n):
        g = result.gestures[i][0]
        h = result.handedness[i][0]
        lines.append(f'  손 {i+1}:')
        lines.append(f'    제스처  : {g.category_name} ({GESTURE_KR.get(g.category_name, g.category_name)})')
        lines.append(f'    신뢰도  : {g.score:.3f}')
        lines.append(f'    왼/오른 : {h.category_name} ({h.score:.3f})')
        lines.append(f'    랜드마크: {len(result.hand_landmarks[i])}개 관절')
    return '\n'.join(lines)

# 템플릿 이미지로 결과 구조 확인
img = make_hand_template('palm')
mp_img = numpy_to_mp_image(img)
result = recognizer.recognize(mp_img)
print(format_gesture_result(result))

# 제스처 결과가 없는 경우의 안전 처리 패턴
print('\n--- 안전 처리 패턴 ---')
if result.hand_landmarks:
    top = result.gestures[0][0]
    name = top.category_name
    print(f'인식된 제스처: {name}')
else:
    print('손이 검출되지 않았습니다. 제스처: None')


---
## 5. 웹캠 실시간 제스처 인식

웹캠에서 실시간으로 제스처를 인식하고 화면에 라벨을 표시합니다.

**주요 단계:**
1. `RunningMode.VIDEO`로 인식기 재생성 (실시간 프레임 처리)
2. 프레임마다 `recognizer.recognize_for_video()` 호출
3. 인식된 제스처 이름과 신뢰도를 프레임에 오버레이
4. 손 랜드마크도 함께 그리기

> ⚠️ 웹캠이 없는 환경에서는 코드만 읽고 구조를 이해하세요.
> `cv2.VideoCapture(0)`가 실패하면 자동으로 종료됩니다.


In [ ]:
# 웹캠 실시간 제스처 인식
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import time

# VIDEO 모드용 인식기 재생성
video_options = vision.GestureRecognizerOptions(
    base_options=python.BaseOptions(model_asset_path=MODEL_PATH),
    running_mode=vision.RunningMode.VIDEO,
    num_hands=2,
    min_hand_detection_confidence=0.5,
    min_hand_presence_confidence=0.5,
    min_tracking_confidence=0.5,
)
video_recognizer = vision.GestureRecognizer.create_from_options(video_options)
print('✓ VIDEO 모드 GestureRecognizer 생성 완료')

# 손 랜드마크 그리기 함수
def draw_hand_landmarks(frame, landmarks, handed_color=(0, 255, 0)):
    """21개 손 랜드마크를 프레임에 그리기"""
    h, w = frame.shape[:2]
    # 랜드마크 연결关系 (MediaPipe 표준)
    connections = [
        (0,1),(1,2),(2,3),(3,4),            # 엄지
        (0,5),(5,6),(6,7),(7,8),            # 검지
        (5,9),(9,10),(10,11),(11,12),       # 중지
        (9,13),(13,14),(14,15),(15,16),     # 약지
        (13,17),(17,18),(18,19),(19,20),    # 새끼
        (0,17),                             # 손바닥
    ]
    pts = [(int(lm.x * w), int(lm.y * h)) for lm in landmarks]
    # 연결선
    for a, b in connections:
        cv2.line(frame, pts[a], pts[b], handed_color, 2)
    # 관절점
    for px, py in pts:
        cv2.circle(frame, (px, py), 4, (0, 0, 255), -1)

# 제스처 라벨 오버레이 함수
def draw_gesture_label(frame, gesture_name, score, handed, x, y):
    """화면 좌상단에 제스처 라벨 박스 그리기"""
    kr = GESTURE_KR.get(gesture_name, gesture_name)
    emoji = GESTURE_EMOJI.get(gesture_name, '❔')
    text = f'{emoji} {gesture_name} ({kr})  {score:.2f}  [{handed}]'
    # 배경 박스
    (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.7, 2)
    cv2.rectangle(frame, (x-5, y-th-10), (x+tw+5, y+5), (50, 50, 50), -1)
    cv2.putText(frame, text, (x, y), cv2.FONT_HERSHEY_SIMPLEX,
                0.7, (255, 255, 255), 2, cv2.LINE_AA)

# ── 웹캡 루프 ──
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    print('✗ 웹캠을 열 수 없습니다. 웹캠 코드를 건너뜁니다.')
    print('  → 코드 구조를 참고용으로만 확인하세요.')
else:
    print('✓ 웹캠 연결됨. "q" 키로 종료.')
    prev_ts = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.flip(frame, 1)  # 거울 모드

        # 타임스탬프 (ms) — VIDEO 모드는 단조 증가 타임스탬프 필요
        ts_ms = int(time.time() * 1000)
        if ts_ms <= prev_ts:
            ts_ms = prev_ts + 1
        prev_ts = ts_ms

        # MediaPipe 추론
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_img = mp.Image(mp.ImageFormat.SRGB, rgb)
        result = video_recognizer.recognize_for_video(mp_img, ts_ms)

        # 결과 시각화
        n = len(result.hand_landmarks)
        y_offset = 40
        for i in range(n):
            g = result.gestures[i][0]
            h = result.handedness[i][0]
            draw_hand_landmarks(frame, result.hand_landmarks[i])
            draw_gesture_label(frame, g.category_name, g.score,
                               h.category_name, 20, y_offset)
            y_offset += 45

        if n == 0:
            cv2.putText(frame, '손을 보여주세요', (20, 40),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (100, 100, 100), 2)

        # 프레임 카운터
        cv2.putText(frame, f'hands: {n}', (frame.shape[1]-120, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

        cv2.imshow('Gesture Recognition (q to quit)', frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()
    print('웹캠 세션 종료.')


---
## 6. 실전: 제스처 기반 간단 제어

인식된 제스처로 **간단한 명령**을 수행합니다.

| 제스처 | 명령 | 동작 |
|--------|------|------|
| `Thumb_Up` 👍 | **YES** | 녹색 표시 + "승인" |
| `Thumb_Down` 👎 | **NO** | 빨간색 표시 + "거절" |
| `Open_Palm` 🖐️ | **STOP** | 노란색 표시 + "정지" |
| `Closed_Fist` ✊ | **PAUSE** | 회색 표시 + "일시정지" |
| `Victory` ✌️ | **WIN** | 파란색 표시 + "승리" |
| 그 외 | 대기 | 흰색 표시 |

> 이 패턴은 프레젠테이션 제어, 게임 입력, 무인 키오스크 등에 응용할 수 있습니다.


In [ ]:
# 제스처 → 명령 매핑
GESTURE_COMMANDS = {
    'Thumb_Up':   {'cmd': 'YES',   'kr': '승인',   'color': (0, 200, 0),     'emoji': '👍'},
    'Thumb_Down': {'cmd': 'NO',    'kr': '거절',   'color': (0, 0, 200),     'emoji': '👎'},
    'Open_Palm':  {'cmd': 'STOP',  'kr': '정지',   'color': (0, 200, 200),   'emoji': '🖐️'},
    'Closed_Fist':{'cmd': 'PAUSE', 'kr': '일시정지','color': (128, 128, 128), 'emoji': '✊'},
    'Victory':    {'cmd': 'WIN',   'kr': '승리',   'color': (200, 0, 0),     'emoji': '✌️'},
    'ILoveYou':   {'cmd': 'LOVE',  'kr': '사랑',   'color': (200, 100, 200), 'emoji': '🤟'},
    'Pointing_Up':{'cmd': 'UP',    'kr': '위로',   'color': (200, 200, 0),   'emoji': '☝️'},
    'None':       {'cmd': 'IDLE',  'kr': '대기',   'color': (255, 255, 255), 'emoji': '❔'},
}

def gesture_to_command(gesture_name, score, min_confidence=0.6):
    """제스처 이름 → 명령 객체. 신뢰도가 낮으면 IDLE."""
    if score < min_confidence:
        return GESTURE_COMMANDS['None']
    return GESTURE_COMMANDS.get(gesture_name, GESTURE_COMMANDS['None'])

# 매핑 테스트 (시뮬레이션)
print('제스처 → 명령 매핑 테스트:')
print('=' * 50)
test_cases = [
    ('Thumb_Up', 0.92),
    ('Thumb_Down', 0.88),
    ('Open_Palm', 0.75),
    ('Victory', 0.81),
    ('Closed_Fist', 0.70),
    ('Thumb_Up', 0.40),  # 낮은 신뢰도 → IDLE
]
for g, s in test_cases:
    cmd = gesture_to_command(g, s)
    print(f'  {GESTURE_EMOJI.get(g,"❔")} {g:<13} (conf={s:.2f}) → '
          f'{cmd["emoji"]} {cmd["cmd"]:<6} ({cmd["kr"]})')


In [ ]:
# 실시간 제스처 제어 루프
import cv2, time
import mediapipe as mp

cap = cv2.VideoCapture(0)
if not cap.isOpened():
    print('✗ 웹캠을 열 수 없습니다. 제어 데모를 건너뜁니다.')
    print('  → 매핑 테이블은 위 셀에서 확인할 수 있습니다.')
else:
    print('✓ 제스처 제어 데모 시작. "q" 키로 종료.')
    print('  👍=YES  👎=NO  🖐️=STOP  ✊=PAUSE  ✌️=WIN')
    prev_ts = 0
    current_cmd = GESTURE_COMMANDS['None']
    stable_count = 0
    stable_cmd = current_cmd
    last_cmd_name = 'IDLE'

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        frame = cv2.flip(frame, 1)
        h, w = frame.shape[:2]

        ts_ms = int(time.time() * 1000)
        if ts_ms <= prev_ts: ts_ms = prev_ts + 1
        prev_ts = ts_ms

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_img = mp.Image(mp.ImageFormat.SRGB, rgb)
        result = video_recognizer.recognize_for_video(mp_img, ts_ms)

        # 현재 프레임 제스처 → 명령
        if result.hand_landmarks:
            g = result.gestures[0][0]
            current_cmd = gesture_to_command(g.category_name, g.score)
            draw_hand_landmarks(frame, result.hand_landmarks[0])
        else:
            current_cmd = GESTURE_COMMANDS['None']

        # 안정화: 같은 명령이 5프레임 연속이어야 확정
        if current_cmd['cmd'] == stable_cmd['cmd']:
            stable_count += 1
        else:
            stable_cmd = current_cmd
            stable_count = 1

        if stable_count >= 5 and stable_cmd['cmd'] != last_cmd_name:
            last_cmd_name = stable_cmd['cmd']
            print(f'  ▶ 명령 확정: {stable_cmd["emoji"]} {stable_cmd["cmd"]} ({stable_cmd["kr"]})')

        # 큰 상태 표시 (화면 중앙)
        cmd = stable_cmd
        overlay = frame.copy()
        cv2.rectangle(overlay, (0, h-120), (w, h), (30, 30, 30), -1)
        cv2.addWeighted(overlay, 0.6, frame, 0.4, 0, frame)

        text = f'{cmd["emoji"]} {cmd["cmd"]} - {cmd["kr"]}'
        cv2.putText(frame, text, (20, h-50), cv2.FONT_HERSHEY_SIMPLEX,
                    1.2, cmd['color'], 3, cv2.LINE_AA)
        cv2.putText(frame, f'confidence-stable: {stable_count} frames',
                    (20, h-15), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200, 200, 200), 1)

        cv2.imshow('Gesture Control (q to quit)', frame)
        if cv2.waitKey(1) & 0xFF == ord('q'): break

    cap.release()
    cv2.destroyAllWindows()
    print('제스처 제어 데모 종료.')


---
## 🎯 정리

| 개념 | 설명 |
|------|------|
| **GestureRecognizer** | 손 랜드마크 기반 제스처 분류 태스크 |
| **모델 파일** | `models/gesture_recognizer.task` (~8 MB) |
| **실행 모드** | `IMAGE`(사진), `VIDEO`(비디오/실시간 프레임), `LIVE_STREAM`(콜백) |
| **recognize()** | IMAGE 모드 추론 (단일 이미지) |
| **recognize_for_video()** | VIDEO 모드 추론 (타임스탬프 필수) |
| **result.gestures** | 손별 제스처 Category 리스트 (top-1) |
| **result.handedness** | 손별 Left/Right 분류 |
| **result.hand_landmarks** | 21개 손 관절 정규화 좌표 |
| **인식 제스처** | 7종: Closed_Fist, Open_Palm, Pointing_Up, Thumb_Down, Thumb_Up, Victory, ILoveYou |
| **제스처 제어** | 제스처 → 명령 매핑 + 안정화 카운터로 노이즈 제거 |

| 메서드 | 용도 |
|--------|------|
| `create_from_options(options)` | 인식기 객체 생성 |
| `recognize(mp_image)` | IMAGE 모드 추론 |
| `recognize_for_video(mp_image, ts_ms)` | VIDEO 모드 추론 (ts 단조 증가) |
| `recognize_async(mp_image, ts_ms, callback)` | LIVE_STREAM 모드 추론 |


---
## 🎯 연습 문제

1. **제스처 카운터 만들기**: 웹캠에서 `Victory` 제스처가 검출될 때마다 카운트를 1씩 증가시키고, 화면에 "✌️ 승리 횟수: N"을 표시하는 코드를 작성하세요. (힌트: 이전 프레임의 제스처를 저장하여 엣지 검출)

2. **제스처 시퀀스 명령**: `Thumb_Up` → `Victory` 순서로 2초 안에 연속 인식되면 "미션 성공" 메시지를 띄우는 시퀀스 인식기를 만들어 보세요. (힌트: 제스처 히스토리와 타임스탬프를 함께 저장)

3. **LIVE_STREAM 모드**: `RunningMode.LIVE_STREAM`과 `recognize_async()`를 사용해 콜백 기반 실시간 인식 코드를 작성하고, VIDEO 모드와의 차이점(비동기 처리, 콜백 함수)을 설명하세요.

4. **제스처 그리기 도구**: `Pointing_Up` 제스처로 손끝 위치를 추적하여 화면에 선을 그리는 간단한 "공중 그리기" 도구를 구현해 보세요. (힌트: `hand_landmarks[8]`이 검지 끝점, 이전 좌표와 현재 좌표를 `cv2.line`으로 연결)
